# Olist Delivery Performance - Project Plan and Runbook

**Big Data Module 2 | Group 7**

> **Where and when are deliveries late, and what does it cost us?**

This notebook is the executable form of the two project documents:

| Document | This notebook |
|---|---|
| `docs/Module2_Group7_Development_Plan.pdf` | Part 1: what we are building and why (sections 1-7) |
| `docs/Module2_Group7_Runbook.pdf` | Part 2: how to run it (sections 8-12) |

The PDFs are what you read. This is what you **run**: every claim the plan makes
about the implementation is checked by a cell, and every setup step verifies
itself instead of asking you to eyeball command output.

### How to read the code cells

| Tag | Meaning |
|---|---|
| `[SAFE]` | Read-only. Inspects state, changes nothing. Run freely. |
| `[WRITES]` | Creates cloud resources or loads data. Read it before running; guarded by a flag that defaults to off. |
| `[TERMINAL]` | Interactive or long-running. Copy into a real terminal; a notebook is the wrong place for it. |

Run this first so every later cell can find the repository.

In [1]:
# [SAFE] Locate the repository root and set up the shared helpers.
import os, pathlib, subprocess, shutil, sys, json, re

ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "Makefile").exists() and (p / "dbt_project").exists()), None)
assert ROOT, "Could not find the repository root. Open this notebook from inside the repo."
os.chdir(ROOT)
print("repository root:", ROOT)


def sh(cmd, quiet=False):
    """Run a shell command, return (exit_code, stdout).

    stdout ONLY. gcloud writes warnings to stderr, and merging the streams made
    an empty result look non-empty, which turned "no account is logged in" into
    a false PASS.
    """
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if not quiet:
        print((r.stdout + r.stderr).strip() or f"(no output, exit {r.returncode})")
    return r.returncode, r.stdout.strip()


def report(rows):
    """Print a PASS/FAIL table. Returns True when everything passed."""
    w = max(len(r[0]) for r in rows) + 2
    for name, ok, detail in rows:
        print(f"{'PASS' if ok else 'FAIL'}  {name:<{w}} {detail}")
    failed = [r[0] for r in rows if not r[1]]
    print()
    print(f"{len(rows) - len(failed)}/{len(rows)} checks passed"
          + (f"  |  outstanding: {', '.join(failed)}" if failed else "  |  all good"))
    return not failed


def load_env():
    """Parse .env without requiring python-dotenv."""
    out = {}
    f = ROOT / ".env"
    if f.exists():
        for line in f.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                out[k.strip()] = v.strip()
    return out


env = load_env()
PROJECT = env.get("GCP_PROJECT_ID", "<PROJECT_ID>")
DEV = env.get("DBT_DEV_DATASET", "")
print("GCP project:", PROJECT)

repository root: /Users/timkoo/DSAI/bigData-module2-project-group7
GCP project: olist-group7


---
---

# PART 1 - The Plan

*Source: `docs/Module2_Group7_Development_Plan.pdf`*

---

## 1. The business case

> **Business Case 3 - Delivery Performance.** Management question: where and when
> are deliveries late?

The course guidance is explicit that a strong project picks one defensible business
case early and lets it determine the grain, the dimensions, the tests and the final
chart. This is ours, and every design decision below is traceable to it.

It decomposes into four questions we must answer with evidence:

| # | Question | Why an executive cares |
|---|---|---|
| D1 | **WHERE** are deliveries late? Which customer states, which seller states, and which routes between them? | Determines which carrier contracts, regional hubs or seller SLAs to renegotiate first. |
| D2 | **WHEN** are deliveries late? Which months, and is the trend improving or worsening? | Determines seasonal capacity planning and whether past logistics investment worked. |
| D3 | **WHAT** does lateness cost? Its association with review score, and the revenue exposed. | Converts a logistics metric into money and satisfaction, which is what a CFO or COO acts on. |
| D4 | **HOW BAD** is the estimate itself? Is the promised date systematically wrong, and by how much? | A promise nobody can keep destroys trust even when delivery time is acceptable. Fixing the promise is often cheaper than fixing the logistics. |

Section 5 of the assignment brief separately mandates monthly sales trends,
top-selling products and customer segmentation. Those are delivered in full, and each
is framed through the delivery lens so the report tells one story rather than four.

### Grading logic this project is built around

- Marks come from the **evidence chain**, not the tool count: source and contract,
  raw load, transformation lineage, quality gates, optimised query, insight,
  recommendation, caveat.
- Roughly **80% of effort** goes to pipeline, architecture, access control, quality
  and reproducibility. Roughly 20% goes to a small number of decisive analyses.
- **Complexity earns credit only when it solves a stated constraint.** Kafka, Spark
  and Redis are deliberately out of scope, with the threshold that would justify each
  written down.
- Every table has a declared **grain**, a primary key, an owner and a description
  before any SQL is written.

---
## 2. Locked decisions

Settled. Changing any of these after Phase 0 is expensive, in particular the BigQuery
location, which cannot be altered once a dataset exists.

| Decision | Choice | Rationale, and what it beat |
|---|---|---|
| **Dataset** | Olist Brazilian E-Commerce | The only one of the three permitted datasets with delivery milestones, review scores, money and geography on both sides of a shipment. **Instacart** has 34M rows but no money columns, so no revenue KPI and the mandated sales analysis is impossible. **London Bicycles** forces an EU-only project and has no customer entity, making the mandated segmentation impossible. |
| **Warehouse** | BigQuery, **US multi-region** | The taught stack. Olist has no residency constraint, so US gives the widest compatibility and keeps the always-free tier (1 TB scanned + 10 GB stored per month). ~120 MB of data means expected spend is zero. Location pinned before any dataset was created because it is immutable. |
| **Ingestion** | Python script + explicit manifest | Olist is a static CSV export, so Meltano's incremental state, catalog and bookmarks would be ceremony around a problem we do not have. A loader with a checksum manifest and `WRITE_TRUNCATE` gives idempotency plus a reconciliation report in less code. Meltano is the documented migration path if a live source appears. |
| **Transformation** | dbt-core (ELT, not ETL) | Raw stays immutable and replayable; transformation is version-controlled SQL and YAML reviewable in a pull request; lineage, tests and docs generate from the same models. |
| **Quality** | dbt tests **+** Great Expectations | A real division of labour, not tool collecting: only GX can reject a malformed file *before* it reaches BigQuery. dbt owns everything already in a table. |
| **Orchestration** | Dagster + GitHub Actions | Optional item 6, taken because it is where operational evidence comes from: asset lineage, retries only where idempotent, and a demonstrable fail-stop. A cron job that runs everything regardless of failure is not orchestration. |
| **Analysis** | SQLAlchemy + pandas, plus a Streamlit dashboard | Required verbatim by the brief. Notebooks and dashboard read the marts layer only. |

### Deliberately out of scope

| Technology | Why not now | What would trigger adopting it |
|---|---|---|
| Apache Spark | 1.55M rows / ~120 MB fit comfortably in BigQuery and in single-node memory. A cluster adds scheduling, shuffle and operations cost for no gain. | Working set exceeds single-node memory, or a job misses its SLA with the scan already optimised. |
| Kafka + Structured Streaming | A static historical export. No event stream, and no business value that decays with latency. | A live order feed exists and a decision (fraud hold, courier dispatch) loses value within minutes. |
| Redis | No low-latency serving surface. A cache is not a source of truth. | A customer-facing API needs sub-10ms reads of a precomputed segment. |
| Meltano | Static CSV has no incremental state to manage. | A continuously growing API or database replica is added as a second source. |

---
## 3. Target architecture

Data flows in one direction. Each numbered step is something a person can rerun on
its own, and each **GATE** is a place where bad data stops rather than continuing
downstream.

| Step | Stage | What happens |
|---|---|---|
| 1 | SOURCE | Kaggle Olist export: nine CSV files, ~120 MB, 1,550,922 rows. |
| 2 | INGEST | `ingestion/load_olist.py` builds a manifest with the sha256, byte size and true CSV record count of every file. |
| 3 | LANDING | GCS `gs://<PROJECT_ID>-olist-raw/olist/raw/v1/` - an immutable, replayable copy of exactly what was downloaded. |
| **-** | **GATE 1** | **Great Expectations, on the files, before the load.** Files present, column set and order, row count vs manifest, non-null keys, money ranges, categorical domains. **On failure nothing is loaded at all.** |
| 4 | RAW | BigQuery `olist_raw.*` with explicit schemas, autodetect disabled, source fields preserved as-is, plus `_batch_id`, `_loaded_at`, `_source_uri`, `_source_sha256`. |
| 5 | STAGING | dbt `stg_*`: rename, cast, deduplicate, translate categories, collapse geolocation to one row per zip prefix. |
| 6 | INTERMEDIATE | dbt `int_*`: the fan-out controls. Payments aggregated to order grain, reviews reduced to one per order, item lines rolled up, delivery milestone arithmetic. |
| 7 | MARTS | dbt `dim_*`, `fct_*`, `agg_*`: three facts, four dimensions, five aggregates, partitioned and clustered. |
| **-** | **GATE 2** | **`dbt build`.** 161 tests interleaved with the models in dependency order. **On failure the marts do not publish.** |
| 8 | SERVING | Notebooks, Streamlit dashboard, report and deck - all reading the marts layer only. |
| - | ACROSS ALL | Dagster asset DAG with a daily schedule runs 1-7 in dependency order and stops downstream work when a gate fails. GitHub Actions checks every pull request. |

### Why four datasets rather than one

`olist_raw` stays immutable so a defect remains reproducible without contaminating
what analysts read; `olist_staging` holds work in progress; `olist_marts` is the only
thing notebooks and the dashboard read; `olist_snapshots` is isolated because dbt
manages its history differently. It also lets the Analytics Owner have read-only
access to marts alone.

In [2]:
# [SAFE] Does the implementation match the architecture described above?
# The plan claims specific model counts and layers. Verify them from the repo.
import glob

stg = sorted(pathlib.Path("dbt_project/models/staging").glob("stg_*.sql"))
itm = sorted(pathlib.Path("dbt_project/models/intermediate").glob("int_*.sql"))
dim = sorted(pathlib.Path("dbt_project/models/marts").glob("dim_*.sql"))
fct = sorted(pathlib.Path("dbt_project/models/marts").glob("fct_*.sql"))
agg = sorted(pathlib.Path("dbt_project/models/marts").glob("agg_*.sql"))
snp = sorted(pathlib.Path("dbt_project/snapshots").glob("*.sql"))
sing = sorted(pathlib.Path("dbt_project/tests").glob("assert_*.sql"))

report([
    ("staging models",        len(stg) == 8,  f"{len(stg)}: " + ", ".join(p.stem[4:] for p in stg)),
    ("intermediate models",   len(itm) == 4,  f"{len(itm)}: " + ", ".join(p.stem[4:] for p in itm)),
    ("dimensions (plan: 4)",  len(dim) == 4,  ", ".join(p.stem for p in dim)),
    ("facts (plan: 3)",       len(fct) == 3,  ", ".join(p.stem for p in fct)),
    ("aggregates (plan: 5)",  len(agg) == 5,  ", ".join(p.stem for p in agg)),
    ("snapshot (plan: 1)",    len(snp) == 1,  ", ".join(p.stem for p in snp)),
    ("singular tests",        len(sing) >= 8, f"{len(sing)} assert_*.sql files"),
    ("ingestion loader",      pathlib.Path("ingestion/load_olist.py").exists(), "ingestion/load_olist.py"),
    ("GX pre-load gate",      pathlib.Path("quality/run_raw_gate.py").exists(), "quality/run_raw_gate.py"),
    ("Dagster definitions",   pathlib.Path("orchestration/definitions.py").exists(), "orchestration/definitions.py"),
    ("Streamlit dashboard",   pathlib.Path("dashboard/app.py").exists(), "dashboard/app.py"),
    ("shared warehouse.py",   pathlib.Path("warehouse.py").exists(), "imported by notebooks AND dashboard"),
])

PASS  staging models         8: customers, geolocation, order_items, order_payments, order_reviews, orders, products, sellers
PASS  intermediate models    4: order_delivery, order_items_rollup, order_payments, order_reviews
PASS  dimensions (plan: 4)   dim_customer, dim_date, dim_product, dim_seller
PASS  facts (plan: 3)        fct_order_items, fct_orders, fct_payments
PASS  aggregates (plan: 5)   agg_customer_rfm, agg_delivery_by_route, agg_delivery_monthly, agg_monthly_sales, agg_product_performance
PASS  snapshot (plan: 1)     snap_product_category
PASS  singular tests         10 assert_*.sql files
PASS  ingestion loader       ingestion/load_olist.py
PASS  GX pre-load gate       quality/run_raw_gate.py
PASS  Dagster definitions    orchestration/definitions.py
PASS  Streamlit dashboard    dashboard/app.py
PASS  shared warehouse.py    imported by notebooks AND dashboard

12/12 checks passed  |  all good


True

---
## 4. Data model: the star schema

> **GRAIN FIRST.** Write down what one row of each fact table represents before
> writing any SQL. The grain determines the primary key, which measures are additive,
> and which joins are safe. Every documented Olist failure comes from skipping this.

Because the business case is delivery performance, **`fct_orders` is the headline
fact**: lateness is a property of a shipment, not of a line within it.
`fct_order_items` remains the revenue backbone, so money questions and delivery
questions are answerable from the same warehouse without mixing grains.

| Model | One row represents | Key |
|---|---|---|
| **`fct_orders`** (primary) | one **order**, as an accumulating snapshot across its milestones | `order_id` |
| `fct_order_items` | one **item line** within an order | `order_id \|\| '-' \|\| order_item_id` |
| `fct_payments` | one **payment record** | `order_id \|\| '-' \|\| payment_sequential` |
| `dim_customer` | one **person** (`customer_unique_id`) | `customer_key` |
| `dim_product` | one product | `product_key` |
| `dim_seller` | one seller | `seller_key` |
| `dim_date` | one calendar day, 2016-01-01 to 2018-12-31 | `date_key` |

**Delivery measures on `fct_orders`:** `approval_days`, `seller_handover_days`,
`carrier_transit_days`, `total_delivery_days`, `promised_delivery_days`,
`delay_vs_promise_days`, `promise_slack_days`, `is_late`, `lateness_bucket`.

### The four fan-out traps

These are the specific, documented ways this dataset silently produces wrong numbers.
Each has a countermeasure in a named model with a uniqueness test.

| Trap | What goes wrong | Countermeasure |
|---|---|---|
| **Payment fan-out** | Joining raw payments to order items multiplies rows: N items x M payment lines. Revenue inflates by a plausible-looking factor. | `int_order_payments` aggregates to order grain before any join. Enforced by a revenue reconciliation test. |
| **Review fan-out** | `review_id` is not unique and some orders carry several reviews. | `int_order_reviews` keeps the most recent review per order, tie-broken on `review_id`. |
| **Customer identity** | `customer_id` is issued **per order**, not per person. Using it for RFM splits one repeat buyer into several one-time customers. | `dim_customer` keys on `customer_unique_id`. Enforced by `assert_rfm_uses_person_grain`. |
| **Geolocation cardinality** | ~1M rows with many per zip prefix; joining it raw explodes the row count. | `stg_geolocation` collapses to one row per prefix (median coordinate, modal city) before it enriches any dimension. |

### Three decisions that change the headline number

1. **Lateness is measured in whole days on date boundaries.** The promise made to the
   customer is a *date*, so an order delivered at 23:00 on the promised day is on
   time. Using timestamps would manufacture lateness nobody experienced.
2. **Stage durations use hours/24, not DATE_DIFF.** Seller handover is frequently
   under a day; rounding to whole days would erase most of that signal.
3. **`is_late` is NULL, not false, for orders that never arrived.** Counting
   undelivered orders as on-time would understate the late rate.

In [3]:
# [SAFE] Do the models actually implement the four fan-out controls and the
# three grain decisions? Check the SQL, not the intention.
checks = []

payments = pathlib.Path("dbt_project/models/intermediate/int_order_payments.sql").read_text()
checks.append(("payments aggregated to order grain",
               "group by order_id" in payments.lower(),
               "int_order_payments groups by order_id"))

reviews = pathlib.Path("dbt_project/models/intermediate/int_order_reviews.sql").read_text()
checks.append(("reviews deduplicated to one per order",
               "qualify" in reviews.lower() and "row_number" in reviews.lower(),
               "int_order_reviews uses QUALIFY ROW_NUMBER"))

customer = pathlib.Path("dbt_project/models/marts/dim_customer.sql").read_text()
checks.append(("dim_customer keyed on the person",
               "customer_unique_id" in customer or "customer_key" in customer,
               "not the per-order customer_id"))

geo = pathlib.Path("dbt_project/models/staging/stg_geolocation.sql").read_text()
checks.append(("geolocation collapsed per zip prefix",
               "group by" in geo.lower() and "approx_quantiles" in geo.lower(),
               "median coordinate per prefix"))

delivery = pathlib.Path("dbt_project/models/intermediate/int_order_delivery.sql").read_text()
checks.append(("lateness measured on DATE boundaries",
               "date_diff(date(order_delivered_at), date(order_promised_at), day)" in delivery.lower(),
               "date_diff on dates, not timestamps"))
checks.append(("stage durations are fractional days",
               "timestamp_diff" in delivery.lower() and "/ 24.0" in delivery,
               "hours / 24.0, not DATE_DIFF"))
checks.append(("is_late is NULL when undelivered",
               "is null then null" in delivery.lower(),
               "never silently false"))

# the measures the plan names must all exist
measures = ["approval_days", "seller_handover_days", "carrier_transit_days",
            "total_delivery_days", "promised_delivery_days", "delay_vs_promise_days",
            "promise_slack_days", "is_late", "lateness_bucket"]
missing = [m for m in measures if m not in delivery]
checks.append(("all 9 delivery measures present", not missing,
               "missing: " + ", ".join(missing) if missing else "as documented in the plan"))

report(checks)

PASS  payments aggregated to order grain      int_order_payments groups by order_id
PASS  reviews deduplicated to one per order   int_order_reviews uses QUALIFY ROW_NUMBER
PASS  dim_customer keyed on the person        not the per-order customer_id
PASS  geolocation collapsed per zip prefix    median coordinate per prefix
PASS  lateness measured on DATE boundaries    date_diff on dates, not timestamps
PASS  stage durations are fractional days     hours / 24.0, not DATE_DIFF
PASS  is_late is NULL when undelivered        never silently false
PASS  all 9 delivery measures present         as documented in the plan

8/8 checks passed  |  all good


True

---
## 5. Data quality: two gates

Severity is assigned deliberately. **error** blocks publication; **warn** publishes
with an explanation. Nothing fails silently and nothing is silently coerced.

| Gate | Tool | Runs | Blocks |
|---|---|---|---|
| **1** | Great Expectations | on the **files**, before the load | Column set and order, row count vs manifest, non-null keys, money ranges, categorical domains, coordinate bounding box. A malformed export never reaches BigQuery. |
| **2** | dbt, 161 tests | on the **models**, after loading | Uniqueness, not-null, referential integrity, accepted values, numeric ranges, milestone chronology, delivery-measure validity, source-to-mart reconciliation. |

The boundary is not stylistic. **dbt can only test data that is already in the
warehouse**, so by the time a dbt test fails the defect has been loaded. GX runs
against the files on disk, so it can answer "should this be loaded at all?".

### The tests that matter most

These catch the failures that would otherwise be invisible and reach a slide:

| Test | Asserts |
|---|---|
| `assert_revenue_reconciles_staging_to_mart` | Mart revenue equals staging revenue **exactly**, at NUMERIC precision. If any join anywhere fans out, the build fails. |
| `assert_order_count_reconciles` | `fct_orders` has exactly one row per staging order. |
| `assert_no_payment_fanout` | Per-order payment totals match staging. |
| `assert_is_late_matches_delay` | The late flag and the measure behind it can never disagree. |
| `assert_delivered_orders_have_delivery_date` | No delivered order silently drops out of the late-rate denominator. |
| `assert_delivery_milestones_in_order` | Nothing is delivered before it was shipped. |
| `assert_rfm_uses_person_grain` | Segmentation is keyed on the person, not the per-order id. |

One test is **warn**, not error, and that is considered rather than convenient:
`assert_payment_total_matches_order_value`. Olist has legitimate mismatches because
vouchers reduce the amount charged. Making it an error would produce a permanently
red build everyone learns to ignore, which is worse than no test.

> **The fault-injection demonstration is a required deliverable.** A passing suite
> proves little; a suite *seen to fail correctly* proves a lot. Inject a duplicate
> `order_item_key` and a negative price, capture the red build and the offending
> rows, show downstream marts did not build, fix, capture green.

In [4]:
# [SAFE] Confirm the quality gates exist and are wired in as the plan describes.
gx = pathlib.Path("quality/run_raw_gate.py").read_text()
sing = {p.stem for p in pathlib.Path("dbt_project/tests").glob("*.sql")}

required = {
    "assert_revenue_reconciles_staging_to_mart",
    "assert_order_count_reconciles",
    "assert_no_payment_fanout",
    "assert_is_late_matches_delay",
    "assert_delivered_orders_have_delivery_date",
    "assert_delivery_milestones_in_order",
    "assert_rfm_uses_person_grain",
    "assert_payment_total_matches_order_value",
}
missing = sorted(required - sing)

warn_test = pathlib.Path("dbt_project/tests/assert_payment_total_matches_order_value.sql").read_text()

report([
    ("GX gate blocks the load", "return 1" in gx and "BLOCKED" in gx,
     "non-zero exit stops make ingest"),
    ("all 8 named singular tests exist", not missing,
     "missing: " + ", ".join(missing) if missing else f"{len(sing)} test files present"),
    ("voucher test is severity=warn", "severity='warn'" in warn_test,
     "documented exception, not an oversight"),
    ("dbt build is the gate, not dbt run",
     "dbt build" in pathlib.Path("Makefile").read_text(),
     "build interleaves tests with models"),
])

PASS  GX gate blocks the load              non-zero exit stops make ingest
PASS  all 8 named singular tests exist     10 test files present
PASS  voucher test is severity=warn        documented exception, not an oversight
PASS  dbt build is the gate, not dbt run   build interleaves tests with models

4/4 checks passed  |  all good


True

---
## 6. Analysis plan

Analysis A is the primary business case and gets the headline chart. B, C and D are
mandated by Section 5 of the brief and are each framed through the delivery lens.
**A chart without an action is not a finding.**

| Analysis | Definition | Trap avoided |
|---|---|---|
| **A. Delivery: WHERE** | Late rate = share of delivered orders with `delay_vs_promise_days > 0`; median and p90 `total_delivery_days`; by customer state, seller state, and the route pair. Volume floor suppresses noisy states. | Reporting only by destination hides origin-driven problems. A mean lets a few outliers define a state. Including undelivered orders understates the rate. |
| **A. Delivery: WHEN** | Monthly late rate and median days; decomposition into approval / seller handover / carrier transit. | Attributing all delay to the carrier when it is in seller handover. Reading partial edge months as a trend. |
| **A. Delivery: COST** | Mean `review_score` by lateness bucket; the same cut **within** a single state and category as a control; share of revenue on late orders; how wrong the estimate itself is. | Confusing correlation with causation. Comparing regions with structurally different logistics. Treating a wrong promise and a slow delivery as one problem. |
| **B. Monthly sales** | Revenue = `sum(item_price)`; freight is a **separate** series, never folded in. Order count = distinct orders from `fct_orders`. Excludes `canceled` / `unavailable`. Overlaid with the monthly late rate. | Never join payments to items. Never mix order grain with item grain in one aggregate. |
| **C. Top products** | Revenue, units and distinct orders reported **together**, with a minimum-sample floor and English category names. Each category carries its late rate and review score. | Ranking on revenue alone flatters high-price, low-volume categories. |
| **D. Customer RFM** | Aggregate to order grain first, then to `customer_unique_id`. Recency measured against the dataset **snapshot date**, never today. | `customer_id` splits one repeat buyer into several one-time customers. |

### Fixed parameters

Defined once in `dbt_project.yml` so no chart can quietly use a different one:

| Parameter | Value | Why fixed |
|---|---|---|
| `olist_snapshot_date` | `2018-10-17` | The analytical "today". The export ends in 2018; using `CURRENT_DATE` would mark every customer dormant and change results every day they run. |
| `analysis_start_date` / `analysis_end_date` | `2016-09-01` / `2018-10-31` | The window with usable coverage. Sparse edge months are annotated, not silently dropped. |
| `excluded_order_statuses` | `canceled`, `unavailable` | Never fulfilled, so excluded from every revenue and delivery KPI. |
| `min_orders_for_reporting` | `30` | A 100% late rate on three orders is noise, not a finding. |

Every finding uses the same five-part insight card: **Observation** (specific values,
period, baseline), **Interpretation** (evidence and speculation separated),
**Decision** (who does what), **Expected impact** (which KPI moves), **Caveat**
(coverage, causal limits, sample size, bias).

In [5]:
# [SAFE] The plan fixes four parameters. Confirm dbt actually declares them.
proj = pathlib.Path("dbt_project/dbt_project.yml").read_text()
expected = {
    "olist_snapshot_date": "2018-10-17",
    "analysis_start_date": "2016-09-01",
    "analysis_end_date": "2018-10-31",
    "min_orders_for_reporting": "30",
}
rows = []
for key, val in expected.items():
    m = re.search(rf'^\s*{key}:\s*"?([^"\n#]+)"?', proj, re.M)
    found = m.group(1).strip().strip('"') if m else None
    rows.append((f"var {key}", found == val, f"{found} (plan says {val})"))

rows.append(("excluded_order_statuses", "canceled" in proj and "unavailable" in proj,
             "canceled, unavailable"))

nb = sorted(p.name for p in pathlib.Path("notebooks").glob("0[123]_*.ipynb"))
rows.append(("three analysis notebooks", len(nb) == 3, ", ".join(nb)))
report(rows)

PASS  var olist_snapshot_date        2018-10-17 (plan says 2018-10-17)
PASS  var analysis_start_date        2016-09-01 (plan says 2016-09-01)
PASS  var analysis_end_date          2018-10-31 (plan says 2018-10-31)
PASS  var min_orders_for_reporting   30 (plan says 30)
PASS  excluded_order_statuses        canceled, unavailable
PASS  three analysis notebooks       01_profiling.ipynb, 02_delivery_performance.ipynb, 03_business_kpis.ipynb

6/6 checks passed  |  all good


True

---
## 7. Delivery phases and ownership

A phase does not start until the previous phase's Definition of Done is met and
merged. The DoD is written as something you can **check**, not claim.

| Phase | Content | Definition of Done |
|---|---|---|
| 0. Setup | Manual steps 1-5, repo scaffold | Every member can run `bq ls` and `dbt debug` against their own dev dataset |
| 1. Scope and design | Business questions, KPI definitions, grain statements, ADR-001..006 | Metric dictionary and ADRs merged to main |
| 2. Profiling | `01_profiling.ipynb`; the four Olist traps quantified with real numbers | Data dictionary complete; anomaly list written |
| 3. Raw load | `load_olist.py`, schemas, manifest, reconciliation | `make ingest` twice gives identical row counts |
| 4. Staging + intermediate | `stg_*`, the fan-out controls | `dbt build --select staging intermediate` green; fan-out check returns zero |
| 5. Marts | dims, facts, aggregates; partitioning and clustering | Full `dbt build` green; ERD matches deployed tables |
| 6. Quality | Full test matrix, GX suite, **fault-injection demo** | Injected fault shows red, the fix shows green, both captured |
| 7. Orchestration + CI | Dagster assets and schedule; GitHub Actions | A failed upstream asset visibly skips downstream; green CI on a PR |
| 8. Analysis | Delivery analysis, business KPIs, dashboard, cost benchmark | Notebooks read from marts via SQLAlchemy, no manual CSV |
| 9. Docs + deck | Diagrams, report, 10-minute deck | Report answers *why these tools*, *why this schema*, *what could go wrong* |
| 10. Dry run | Clean-clone reproduction, timed rehearsal | A teammate reproduces from an empty folder using only the README |

### Ownership and access control

Five or more people on one GCP project is the likeliest source of accidental damage.

| Role | Owns | Access |
|---|---|---|
| Platform / Ingestion | GCP project, GCS bucket, `olist_raw`, the loader | `bigquery.jobUser` + `storage.admin`; Data Editor on `olist_raw` **only** |
| Modeling | dbt models, star schema, ERD | Data Editor on staging, marts, snapshots |
| Quality | dbt tests, GX suites, fault-injection demo | Viewer everywhere; Editor on own dev dataset |
| Orchestration / DevOps | Dagster, GitHub Actions, secrets, Makefile | Owns the least-privilege CI service account |
| Analytics / Comms | Notebooks, dashboard, metric dictionary, report, deck | **Viewer on `olist_marts` only** |

**Non-negotiable rules.** One person owns raw uploads. Everyone develops into their
own `dbt_<name>` dataset and promotes only through a passing build. Local access is
**user OAuth**, so every BigQuery job is attributable to a person; no service-account
key is ever downloaded to a laptop. The single service account is for CI, and cannot
touch `olist_raw`.

---
---

# PART 2 - The Runbook

*Source: `docs/Module2_Group7_Runbook.pdf`*

Steps 1-5 are done **once**. After that only Steps 6 and 7 are needed.

| Step | What | Who | How often |
|---|---|---|---|
| 1 | Python environment | Every member | Once per machine |
| 2 | GCP project, billing, APIs, team access | **Platform Owner only** | Once for the team |
| 3 | Authenticate `gcloud` locally | Every member | Once per machine |
| 4 | Download the Olist dataset | Whoever runs the load | Once |
| 5 | Bucket, datasets, config | Platform Owner (shared) + everyone (own dev dataset) | Once |
| 6 | Run the pipeline: `make all` | Anyone | Whenever data or models change |
| 7 | Open the dashboard: `make dashboard` | Anyone | Any time |

### Two defects that were fixed for the first run

Both would have stopped the very first run. Already corrected in the repository; this
note exists so the team understands why the Makefile looks the way it does.

1. **dbt could not see `.env`.** `python-dotenv` loads it for the Python scripts, but
   dbt resolves `env_var()` against the real process environment, so every dbt target
   failed with *"Env var required but not provided: GCP_PROJECT_ID"*. The Makefile now
   includes and exports `.env` automatically.
2. **`make all` ran the snapshot before the models.** The snapshot reads
   `stg_products`, a view that does not exist on a fresh warehouse, so the first run
   always failed. It was also redundant: `dbt build` already runs snapshots in
   dependency order. The separate step was removed, and the same fix applied to the
   nightly GitHub Actions workflow.

---
## 8. Preflight - where am I right now?

Run this at any time. It says which of Steps 1-5 are already done, so you can pick up
exactly where you left off.

In [6]:
# [SAFE] Read-only inspection of this machine.
env = load_env()
PROJECT = env.get("GCP_PROJECT_ID", "<PROJECT_ID>")
DEV = env.get("DBT_DEV_DATASET", "")

csvs = sorted((ROOT / "data" / "raw").glob("*.csv")) if (ROOT / "data" / "raw").exists() else []
_, account = sh("gcloud auth list --filter=status:ACTIVE --format='value(account)'", quiet=True)
adc = pathlib.Path.home() / ".config/gcloud/application_default_credentials.json"
profiles = pathlib.Path.home() / ".dbt/profiles.yml"

report([
    ("Step 1  Python is 3.11",   sys.version_info[:2] == (3, 11), f"running {sys.version.split()[0]}"),
    ("Step 1  dbt on PATH",      shutil.which("dbt") is not None, shutil.which("dbt") or "not found"),
    ("Step 1  dbt packages",     (ROOT / "dbt_project/dbt_packages").exists(), "dbt deps has run"),
    ("Step 3  gcloud account",   bool(account), account or "run: gcloud auth login"),
    ("Step 3  ADC credentials",  adc.exists(), "gcloud auth application-default login"),
    ("Step 4  9 source CSVs",    len(csvs) == 9, f"{len(csvs)} file(s) in data/raw/"),
    ("Step 5  .env present",     bool(env), f"{len(env)} variables"),
    ("Step 5  GCP_PROJECT_ID",   bool(env.get("GCP_PROJECT_ID")), env.get("GCP_PROJECT_ID", "not set")),
    ("Step 5  DBT_DEV_DATASET",  bool(DEV), DEV or "not set"),
    ("Step 5  dbt profiles.yml", profiles.exists(), str(profiles)),
])

PASS  Step 1  Python is 3.11     running 3.11.16
PASS  Step 1  dbt on PATH        /Users/timkoo/miniconda3/envs/bd-m2-g7/bin/dbt
PASS  Step 1  dbt packages       dbt deps has run
PASS  Step 3  gcloud account     timothykoo7@gmail.com
PASS  Step 3  ADC credentials    gcloud auth application-default login
PASS  Step 4  9 source CSVs      9 file(s) in data/raw/
PASS  Step 5  .env present       14 variables
PASS  Step 5  GCP_PROJECT_ID     olist-group7
PASS  Step 5  DBT_DEV_DATASET    dbt_TK
PASS  Step 5  dbt profiles.yml   /Users/timkoo/.dbt/profiles.yml

10/10 checks passed  |  all good


True

---
## 9. Step 1 - Python environment

*Every member. About 10 minutes.*

> **Python version matters.** The system default `python3` is **3.14**, which dbt does
> **not** support. The project is pinned to **3.11**. Installing into the wrong
> interpreter is the most common way this setup goes wrong, and the error is confusing.

`[TERMINAL]` - `conda activate` cannot change the kernel of a running notebook, so run
these in a terminal, then reopen this notebook on the new kernel:

```bash
conda create -y -n bd-m2-g7 python=3.11
conda activate bd-m2-g7
make setup
```

`make setup` installs `requirements.txt` and runs `dbt deps`.

In [7]:
# [SAFE] Verify the interpreter and toolchain THIS kernel is using.
rows = [("Python is 3.11", sys.version_info[:2] == (3, 11), sys.version.split()[0])]
for tool in ("dbt", "gcloud", "bq", "streamlit", "make"):
    rows.append((f"{tool} on PATH", shutil.which(tool) is not None, shutil.which(tool) or "missing"))
for mod in ("google.cloud.bigquery", "great_expectations", "sqlalchemy", "pandas", "dagster"):
    try:
        __import__(mod); ok, detail = True, "importable"
    except Exception as e:
        ok, detail = False, type(e).__name__
    rows.append((f"import {mod}", ok, detail))
report(rows)

/Users/timkoo/miniconda3/envs/bd-m2-g7/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.6.0)/charset_normalizer (3.5.1) doesn't match a supported version!
  warnings.warn(


PASS  Python is 3.11                 3.11.16
PASS  dbt on PATH                    /Users/timkoo/miniconda3/envs/bd-m2-g7/bin/dbt
PASS  gcloud on PATH                 /opt/homebrew/bin/gcloud
PASS  bq on PATH                     /opt/homebrew/bin/bq
PASS  streamlit on PATH              /Users/timkoo/miniconda3/envs/bd-m2-g7/bin/streamlit
PASS  make on PATH                   /usr/bin/make
PASS  import google.cloud.bigquery   importable
PASS  import great_expectations      importable
PASS  import sqlalchemy              importable
PASS  import pandas                  importable
PASS  import dagster                 importable

11/11 checks passed  |  all good


True

---
## 10. Step 2 - Create the GCP project

*Platform Owner only, once for the team. About 15 minutes. Browser work.*

1. Open <https://console.cloud.google.com> and sign in with the account that will own
   the project. Use one you will still control at submission time.
2. **Enable billing.** Billing -> *Link a billing account*. A new account gets a
   300 USD / 90-day trial. Billing must be on for the BigQuery API to work at all, but
   the always-free tier (1 TB scanned + 10 GB stored per month) covers this project
   many times over. **Expected real spend: zero.**
3. **Create the project.** Project selector -> *New Project* -> name it
   `bigdata-m2-group7`. Google appends digits to make a globally unique **Project ID**
   (e.g. `bigdata-m2-group7-473012`). Copy that ID; it is *not* the display name.
4. **Enable the APIs.** APIs and Services -> *Enable APIs* -> **BigQuery API**, then
   **Cloud Storage API**.
5. **Invite the team.** IAM and Admin -> IAM -> *Grant access* -> add each teammate's
   Google email with the role **BigQuery Job User**. Dataset-level roles come in Step 5.

In [8]:
# [SAFE] Confirm the project exists and is active.
rc, out = sh(f"gcloud projects describe {PROJECT} --format='value(projectId,lifecycleState)'")
print()
print("Project is ACTIVE." if rc == 0 and "ACTIVE" in out
      else "Not reachable yet. Finish Step 2, or check the Project ID in .env.")

olist-group7	ACTIVE

Project is ACTIVE.


---
## 11. Step 3 - Authenticate locally

*Every member, on their own machine. About 5 minutes.*

`[TERMINAL]` - the first two open a browser and wait for you:

```bash
gcloud auth login
gcloud auth application-default login
gcloud config set project <PROJECT_ID>
```

The second creates the **Application Default Credentials** that dbt, the Python
client, SQLAlchemy and the dashboard all read. It is a **separate** credential from
the first. Skipping it is the number-one cause of *"could not automatically determine
credentials"*.

> **Never download a service-account key for local work.** User OAuth makes every
> BigQuery job attributable to a person, which is what the course access-control
> guidance requires. The only service account is for GitHub Actions, and its key
> lives solely in GitHub Secrets. If one ever leaks, **revoke it** - deleting the file
> in a later commit does not remove it from git history.

In [9]:
# [SAFE] Verify both credentials and that BigQuery answers.
adc = pathlib.Path.home() / ".config/gcloud/application_default_credentials.json"
_, account = sh("gcloud auth list --filter=status:ACTIVE --format='value(account)'", quiet=True)
rc, out = sh("bq ls --max_results=1", quiet=True)

report([
    ("gcloud account active", bool(account), account or "run: gcloud auth login"),
    ("ADC file exists",       adc.exists(),  "gcloud auth application-default login"),
    ("BigQuery responds",     rc == 0,       (out.splitlines() or ["-"])[0][:70]),
    ("no key file in repo",
     not list(ROOT.glob("*-key.json")) and not list(ROOT.glob("service-account*.json")),
     "local key files must never exist here"),
])

PASS  gcloud account active   timothykoo7@gmail.com
PASS  ADC file exists         gcloud auth application-default login
PASS  BigQuery responds       datasetId    
PASS  no key file in repo     local key files must never exist here

4/4 checks passed  |  all good


True

---
## 12. Step 4 - Download the dataset

*Whoever runs the load. About 5 minutes.*

Open <https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce>, sign in, click
**Download** (~45 MB zipped, ~120 MB unzipped), unzip into `data/raw/`.

Or with the Kaggle CLI, after placing your token at `~/.kaggle/kaggle.json` (chmod 600):

```bash
kaggle datasets download -d olistbr/brazilian-ecommerce -p data/raw --unzip
```

`data/` is git-ignored, so raw CSVs never enter the repository. Record the licence
(CC BY-NC-SA 4.0) and download date in the data dictionary.

In [10]:
# [SAFE] Count the REAL CSV records and compare against the published counts.
# Uses the csv module, not a line count: review text contains embedded newlines, so
# `wc -l` overcounts olist_order_reviews by tens of thousands of lines.
import csv

EXPECTED = {
    "olist_orders_dataset.csv": 99_441,
    "olist_order_items_dataset.csv": 112_650,
    "olist_order_payments_dataset.csv": 103_886,
    "olist_order_reviews_dataset.csv": 99_224,
    "olist_customers_dataset.csv": 99_441,
    "olist_products_dataset.csv": 32_951,
    "olist_sellers_dataset.csv": 3_095,
    "olist_geolocation_dataset.csv": 1_000_163,
    "product_category_name_translation.csv": 71,
}
RAW = ROOT / "data" / "raw"

rows = []
for name, expected in EXPECTED.items():
    path = RAW / name
    if not path.exists():
        rows.append((name, False, "missing from data/raw/")); continue
    with path.open(newline="", encoding="utf8") as fh:
        r = csv.reader(fh); next(r, None)
        n = sum(1 for _ in r)
    rows.append((name, n == expected, f"{n:>9,} rows (expected {expected:,})"))
report(rows)

PASS  olist_orders_dataset.csv                   99,441 rows (expected 99,441)
PASS  olist_order_items_dataset.csv             112,650 rows (expected 112,650)
PASS  olist_order_payments_dataset.csv          103,886 rows (expected 103,886)
PASS  olist_order_reviews_dataset.csv            99,224 rows (expected 99,224)
PASS  olist_customers_dataset.csv                99,441 rows (expected 99,441)
PASS  olist_products_dataset.csv                 32,951 rows (expected 32,951)
PASS  olist_sellers_dataset.csv                   3,095 rows (expected 3,095)
PASS  olist_geolocation_dataset.csv           1,000,163 rows (expected 1,000,163)
PASS  product_category_name_translation.csv          71 rows (expected 71)

9/9 checks passed  |  all good


True

---
## 13. Step 5 - Bucket, datasets and config

*Platform Owner creates the shared resources; every member does the config. ~10 min.*

> **IRREVERSIBLE.** A BigQuery dataset's location **cannot be changed after
> creation**, and datasets in different locations cannot be joined. Everything here is
> **US multi-region**. The whole team must agree before this runs; fixing it later
> means deleting and reloading everything.

### 13.1 Shared resources - Platform Owner only

In [11]:
# [WRITES] Creates cloud resources. Run ONCE, by the Platform Owner only.
# Everyone else: skip this cell and go to 13.2.
CREATE_SHARED = False   # set True when you are the Platform Owner

if CREATE_SHARED:
    sh(f"gcloud storage buckets create gs://{PROJECT}-olist-raw "
       f"--location=US --uniform-bucket-level-access")
    for ds in ("olist_raw", "olist_staging", "olist_marts", "olist_snapshots"):
        sh(f'bq --location=US mk --dataset "{PROJECT}:{ds}"')
else:
    print("Skipped. Set CREATE_SHARED = True if you are the Platform Owner.")
    print(f"Would create: gs://{PROJECT}-olist-raw and 4 datasets in US.")

Skipped. Set CREATE_SHARED = True if you are the Platform Owner.
Would create: gs://olist-group7-olist-raw and 4 datasets in US.


### 13.2 Your own dev dataset - every member

Everyone develops into their own dataset, so nobody can overwrite anyone else's
tables. Only a passing build promotes anything to the shared marts.

In [12]:
# [WRITES] Creates one small empty dataset that only you use.
YOUR_NAME = DEV.replace("dbt_", "") or "yourname"
CREATE_MY_DATASET = False   # set True to create it

if CREATE_MY_DATASET:
    sh(f'bq --location=US mk --dataset "{PROJECT}:dbt_{YOUR_NAME}"')
else:
    print(f"Would create: {PROJECT}:dbt_{YOUR_NAME}")
    print("Set CREATE_MY_DATASET = True to run it.")

Would create: olist-group7:dbt_TK
Set CREATE_MY_DATASET = True to run it.


### 13.3 Config files - every member

```bash
cp .env.example .env
mkdir -p ~/.dbt && cp dbt_project/profiles.yml.example ~/.dbt/profiles.yml
```

Then edit `.env` and fill in at least:

```
GCP_PROJECT_ID=<your Project ID>
GCS_RAW_BUCKET=<PROJECT_ID>-olist-raw
DBT_DEV_DATASET=dbt_<yourname>
```

Both `.env` and `~/.dbt/profiles.yml` are git-ignored. `.env.example` holds variable
names only and is the file that is committed.

In [13]:
# [SAFE] The most important check here: dataset LOCATION, before any data is loaded.
rows = []
for ds in ("olist_raw", "olist_staging", "olist_marts", "olist_snapshots"):
    rc, out = sh(f"bq show --format=prettyjson {PROJECT}:{ds}", quiet=True)
    if rc != 0:
        rows.append((ds, False, "does not exist yet")); continue
    loc = json.loads(out).get("location", "?")
    rows.append((ds, loc.upper() == "US", f"location = {loc}"))

if DEV:
    rc, out = sh(f"bq show --format=prettyjson {PROJECT}:{DEV}", quiet=True)
    loc = json.loads(out).get("location", "?") if rc == 0 else None
    rows.append((DEV, rc == 0 and str(loc).upper() == "US",
                 f"location = {loc}" if rc == 0 else "does not exist yet"))

if not report(rows):
    print()
    print("Any dataset NOT in US must be deleted and recreated NOW, while empty:")
    print(f"  bq rm -r -f -d {PROJECT}:<dataset>")
    print(f"  bq --location=US mk --dataset {PROJECT}:<dataset>")

PASS  olist_raw         location = US
PASS  olist_staging     location = US
PASS  olist_marts       location = US
PASS  olist_snapshots   location = US
FAIL  dbt_TK            does not exist yet

4/5 checks passed  |  outstanding: dbt_TK

Any dataset NOT in US must be deleted and recreated NOW, while empty:
  bq rm -r -f -d olist-group7:<dataset>
  bq --location=US mk --dataset olist-group7:<dataset>


In [14]:
# [SAFE] Confirm dbt can actually connect with your config.
# Use `make debug`, NOT a bare `dbt debug`. The Makefile is the only thing that
# loads and exports .env; a bare dbt command bypasses it and fails with
# "Env var required but not provided: GCP_PROJECT_ID".
sh("make debug")

cd dbt_project && dbt debug --target dev
03:12:58  Running with dbt=1.11.11
03:12:58  dbt version: 1.11.11
03:12:58  python version: 3.11.16
03:12:58  python path: /Users/timkoo/miniconda3/envs/bd-m2-g7/bin/python3.11
03:12:58  os info: macOS-26.6.2-arm64-arm-64bit
03:13:00  Using profiles dir at /Users/timkoo/.dbt
03:13:00  Using profiles.yml file at /Users/timkoo/.dbt/profiles.yml
03:13:00  Using dbt_project.yml file at /Users/timkoo/DSAI/bigData-module2-project-group7/dbt_project/dbt_project.yml
03:13:00  adapter type: bigquery
03:13:00  adapter version: 1.10.3
03:13:00  Configuration:
03:13:00    profiles.yml file [OK found and valid]
03:13:00    dbt_project.yml file [OK found and valid]
03:13:00  Required dependencies:
03:13:00   - git [OK found]

03:13:00  Connection:
03:13:00    method: oauth
03:13:00    database: olist-group7
03:13:00    execution_project: olist-group7
03:13:00    schema: dbt_TK
03:13:00    location: US
03:13:00    priority: interactive
03:13:00    maximum_byte

(0,
 'cd dbt_project && dbt debug --target dev\n\x1b03:12:58  Running with dbt=1.11.11\n\x1b03:12:58  dbt version: 1.11.11\n\x1b03:12:58  python version: 3.11.16\n\x1b03:12:58  python path: /Users/timkoo/miniconda3/envs/bd-m2-g7/bin/python3.11\n\x1b03:12:58  os info: macOS-26.6.2-arm64-arm-64bit\n\x1b03:13:00  Using profiles dir at /Users/timkoo/.dbt\n\x1b03:13:00  Using profiles.yml file at /Users/timkoo/.dbt/profiles.yml\n\x1b03:13:00  Using dbt_project.yml file at /Users/timkoo/DSAI/bigData-module2-project-group7/dbt_project/dbt_project.yml\n\x1b03:13:00  adapter type: bigquery\n\x1b03:13:00  adapter version: 1.10.3\n\x1b03:13:00  Configuration:\n\x1b03:13:00    profiles.yml file [\x1bOK found and valid\x1b[0m]\n\x1b03:13:00    dbt_project.yml file [\x1bOK found and valid\x1b[0m]\n\x1b03:13:00  Required dependencies:\n\x1b03:13:00   - git [\x1bOK found\x1b[0m]\n\n\x1b03:13:00  Connection:\n\x1b03:13:00    method: oauth\n\x1b03:13:00    database: olist-group7\n\x1b03:13:00    executi

---
## 14. Step 6 - Run the pipeline

*Anyone, any time. First run about 5 to 10 minutes, mostly uploading and loading.*

`[TERMINAL] recommended` - `make all` streams a lot of output and takes minutes. A
terminal handles that better than a notebook.

```bash
make all
```

Stages run in order and the chain stops at the first failure: `manifest` -> **GATE 1**
-> `ingest` -> **GATE 2** (`dbt build`) -> `unit`.

### Every make target

| Command | What it does |
|---|---|
| `make help` | List every target |
| `make setup` | Install requirements and dbt packages. Once per machine |
| `make manifest` | Checksum and row-count the local CSVs. No cloud calls |
| `make gate` | Quality Gate 1 alone: validate files without loading |
| `make ingest` | Upload to GCS, load BigQuery, reconcile row counts |
| `make build` | Quality Gate 2 alone: `dbt build` (models, snapshot, tests) |
| `make test` | dbt tests only, against what is already built |
| `make unit` | pytest. No credentials needed |
| `make lint` | sqlfluff over all dbt SQL |
| **`make all`** | **The whole pipeline: gate, ingest, build, unit** |
| `make analyse` | Execute the three analysis notebooks |
| **`make dashboard`** | **Serve the interactive dashboard** |
| `make docs` | Generate and serve the dbt lineage graph |
| `make evidence` | Collect test artifacts into `docs/evidence/` |
| `make clean` | Remove dbt build artifacts |

In [15]:
# [WRITES] Uploads to GCS, loads BigQuery, builds all models. Minutes to run.
RUN_PIPELINE = False   # set True to actually run it

if RUN_PIPELINE:
    sh("make all")
else:
    print("Skipped. In a terminal, run:\n")
    print("    make all\n")
    print("Or stage by stage, so a failure is easy to localise:\n")
    for step in ("make manifest", "make gate", "make ingest", "make build", "make unit"):
        print("   ", step)

Skipped. In a terminal, run:

    make all

Or stage by stage, so a failure is easy to localise:

    make manifest
    make gate
    make ingest
    make build
    make unit


### 14.1 Prove idempotency - this is a graded deliverable

Running the pipeline twice must produce **identical** row counts. The loader uses
`WRITE_TRUNCATE` per batch, so a rerun replaces rather than appends. Demonstrating it
is evidence, not decoration.

In [16]:
# [SAFE] Read-only reconciliation of the raw tables against the published counts.
rows = []
for table, expected in [("orders", 99_441), ("order_items", 112_650),
                        ("order_payments", 103_886), ("order_reviews", 99_224),
                        ("customers", 99_441), ("products", 32_951),
                        ("sellers", 3_095), ("geolocation", 1_000_163),
                        ("product_category_translation", 71)]:
    rc, out = sh(f'bq query --use_legacy_sql=false --format=csv '
                 f'"SELECT COUNT(*) FROM \`{PROJECT}.olist_raw.{table}\`"', quiet=True)
    if rc != 0:
        rows.append((table, False, "not loaded yet")); continue
    n = int(out.strip().splitlines()[-1])
    rows.append((table, n == expected, f"{n:>9,} rows (expected {expected:,})"))
report(rows)
print()
print("Run `make ingest` again and re-run this cell: the numbers must be identical.")

PASS  orders                            99,441 rows (expected 99,441)
PASS  order_items                      112,650 rows (expected 112,650)
PASS  order_payments                   103,886 rows (expected 103,886)
PASS  order_reviews                     99,224 rows (expected 99,224)
PASS  customers                         99,441 rows (expected 99,441)
PASS  products                          32,951 rows (expected 32,951)
PASS  sellers                            3,095 rows (expected 3,095)
PASS  geolocation                    1,000,163 rows (expected 1,000,163)
PASS  product_category_translation          71 rows (expected 71)

9/9 checks passed  |  all good

Run `make ingest` again and re-run this cell: the numbers must be identical.


---
## 15. Step 7 - Open the dashboard

`[TERMINAL]` - Streamlit runs a server that blocks until you stop it.

```bash
make dashboard
```

Opens at <http://localhost:8501>. `Ctrl+C` in the terminal to stop.

### The five tabs

| Tab | Answers | What to look at |
|---|---|---|
| **Where** | Which destinations and routes are late? | Late rate by state against the national average; an origin-to-destination region heat map; the routes carrying the most late orders, with the revenue behind each |
| **When** | Which months, and is it improving? | Volume against late rate by purchase month; first-half vs second-half in percentage points; whether days are lost in payment approval, seller handover or carrier transit |
| **What it costs** | What does lateness cost? | Review score by lateness bucket; the same penalty recomputed *within* each state as a control; revenue exposed; whether the promise is padded or optimistic |
| **Business KPIs** | The three metrics the brief requires | Monthly sales with late rate overlaid; a revenue-vs-late-rate scatter surfacing categories that sell well but ship badly; RFM segments with delivery experience |
| **Data quality** | Can these numbers be trusted? | Runs the revenue reconciliation, the order-grain check, the person-grain RFM check and an app-vs-model agreement check **live** against the warehouse |

> **Demonstrate the Data quality tab during the presentation.** It re-runs the
> reconciliations against the live warehouse in front of the audience. A dashboard
> that can prove its own numbers are not inflated by a join is far more persuasive
> than one that simply asserts it.

**Why the dashboard cannot drift from the pipeline**

- It reads the **marts layer only**. No CSV, no raw tables, no local files.
- It **does not redefine metrics**; definitions live in `docs/metric_dictionary.md`
  and are implemented in dbt.
- Interactive filtering *does* force the late rate to be recomputed against
  `fct_orders`, because a pre-aggregated model cannot be sliced by an arbitrary filter
  combination. That SQL is confined to one constant in the app, and the Data quality
  tab asserts live that it still agrees with `agg_delivery_monthly`.
- `warehouse.py` sits at the repository root and is imported by **both** the notebooks
  and the dashboard, so the two cannot read different datasets while both look right.

In [17]:
# [SAFE] Check the dashboard's prerequisites without starting the server.
rc_marts, _ = sh(f"bq ls {PROJECT}:{DEV}_marts", quiet=True) if DEV else (1, "")
report([
    ("streamlit installed", shutil.which("streamlit") is not None,
     shutil.which("streamlit") or "pip install -r requirements.txt"),
    ("dashboard/app.py exists", (ROOT / "dashboard/app.py").exists(), "the app itself"),
    ("warehouse.py at repo root", (ROOT / "warehouse.py").exists(),
     "shared by notebooks and dashboard"),
    ("marts dataset populated", rc_marts == 0,
     "ready" if rc_marts == 0 else "run `make all` first"),
])
print()
print("When all four pass, run this in a TERMINAL:")
print()
print("    make dashboard")

PASS  streamlit installed         /Users/timkoo/miniconda3/envs/bd-m2-g7/bin/streamlit
PASS  dashboard/app.py exists     the app itself
PASS  warehouse.py at repo root   shared by notebooks and dashboard
PASS  marts dataset populated     ready

4/4 checks passed  |  all good

When all four pass, run this in a TERMINAL:

    make dashboard


---
## 16. Optional outputs

```bash
make analyse   # execute all three analysis notebooks headless
make docs      # dbt documentation and the full lineage graph
make lint      # sqlfluff over all dbt SQL
```

Dagster orchestration UI - `[TERMINAL]`, opens at <http://localhost:3000>:

```bash
mkdir -p orchestration/.dagster_home
export DAGSTER_HOME=$PWD/orchestration/.dagster_home
dagster dev -f orchestration/definitions.py
```

Dagster requires DAGSTER_HOME to point at a directory that already exists; it
does not create one for you, and fails with "is not a directory or does not
exist" if the mkdir step above is skipped.

This is where you demonstrate that a failing upstream quality gate **skips** the
downstream assets - the difference between orchestration and a cron job that runs
everything regardless.

---
## 17. Troubleshooting

| Symptom | Cause and fix |
|---|---|
| `Env var required but not provided: GCP_PROJECT_ID` | `.env` missing or incomplete. The Makefile only includes `.env` when it exists. |
| `could not automatically determine credentials` | `gcloud auth application-default login` was skipped. It is separate from `gcloud auth login`. |
| `Dataset olist_raw is in EU, but this project is pinned to US` | Wrong location. The loader asserts this and fails fast on purpose. Delete and recreate with `--location=US` **before** loading. |
| `Missing source file: data/raw/olist_orders_dataset.csv` | Kaggle archive not unzipped into `data/raw/`. See Step 4. |
| Quality Gate 1 reports FAIL and nothing loads | The gate working as designed. The log names the file, column and offending record count. Fix the data. **Do not bypass it.** |
| A dbt test fails and marts do not build | Also by design. The reconciliation tests in particular mean a join has fanned out and a revenue figure would have been wrong. Fix the model, do not lower the test. |
| dbt will not install, or errors on startup | Wrong Python. `python --version` must be 3.11. |
| Dashboard shows "Not configured yet" | `.env` missing or incomplete. It is showing setup instructions rather than a stack trace. |
| Dashboard shows "Could not reach the marts layer" | Credentials fine, but marts do not exist. Run `make all`. |
| Port 8501 already in use | A dashboard is already running, or use `--server.port 8502`. |
| `$DAGSTER_HOME "..." is not a directory or does not exist` | Dagster requires the directory to already exist - it will not create one for you. Run `mkdir -p orchestration/.dagster_home` before the `export`. |
| Dagster: `profiles_dir ... does not contain a profiles.yml file`, thrown from inside `dagster_dbt/dbt_project.py` | A dagster_dbt internal step (`DbtProject.prepare_if_dev()`) builds its own `DbtCliResource` at import time and, unlike the `dbt` CLI itself, does NOT fall back to `DBT_PROFILES_DIR` or `~/.dbt`. `orchestration/definitions.py` already passes `profiles_dir` directly to the `DbtProject(...)` constructor for this reason. If you see this, something reverted that argument. |
| Restarting `dagster dev` keeps showing the pre-fix error | `dagster dev` spawns separate webserver, code-server, grpc and daemon child processes with their own command lines. Killing only the parent (`pkill -f "dagster dev"`) orphans the children, which keep the old, broken code location bound to port 3000. Use `pkill -9 -f dagster` and confirm `lsof -ti :3000` is empty before starting again. |

---
## 18. Acceptance checklist

Run this when you believe the pipeline is working. It **executes** the checks instead
of asking you to tick boxes by hand.

> **The SQL has never run against a real BigQuery warehouse.** Every check made before
> your first run was *static*: dbt and sqlfluff parsed the SQL against the BigQuery
> dialect, and the dashboard was exercised against synthetic data. Semantic
> correctness on the real dataset can only be confirmed here. **If `dbt build` errors
> on the first run, capture the full output before changing anything.**

In [18]:
# [SAFE] The acceptance checklist, executed.
rows = []

rc, out = sh("python -m pytest tests -q", quiet=True)
rows.append(("pytest suite", rc == 0, out.strip().splitlines()[-1] if out else "-"))

rr = ROOT / "dbt_project/target/run_results.json"
if rr.exists():
    res = json.loads(rr.read_text())["results"]
    bad = [r for r in res if r["status"] in ("error", "fail")]
    rows.append(("dbt build: 0 failures", not bad, f"{len(res)} nodes, {len(bad)} failed"))
else:
    rows.append(("dbt build has been run", False, "no target/run_results.json yet"))

if DEV:
    sql = (f"SELECT (SELECT ROUND(SUM(item_price),2) FROM "
           f"`{PROJECT}.{DEV}_staging.stg_order_items`) AS staging, "
           f"(SELECT ROUND(SUM(order_revenue),2) FROM "
           f"`{PROJECT}.{DEV}_marts.fct_orders`) AS mart")
    rc, out = sh(f'bq query --use_legacy_sql=false --format=csv "{sql}"', quiet=True)
    if rc == 0 and len(out.strip().splitlines()) > 1:
        staging, mart = out.strip().splitlines()[-1].split(",")
        rows.append(("revenue reconciles staging = mart", staging == mart,
                     f"{staging} vs {mart}"))
    else:
        rows.append(("revenue reconciles staging = mart", False, "marts not built yet"))
else:
    rows.append(("revenue reconciles staging = mart", False, "DBT_DEV_DATASET not set"))

rc, tracked = sh("git ls-files", quiet=True)
leaks = [f for f in tracked.splitlines()
         if f == ".env" or f.startswith("data/") or f.endswith("-key.json")]
rows.append(("no secrets or data in git", not leaks, ", ".join(leaks) or "clean"))

nbs = sorted((ROOT / "notebooks").glob("0[123]_*.ipynb"))
rows.append(("three analysis notebooks", len(nbs) == 3, f"{len(nbs)} present"))

report(rows)

PASS  pytest suite                        19 passed, 1 warning in 0.49s
PASS  dbt build: 0 failures               187 nodes, 0 failed
FAIL  revenue reconciles staging = mart   marts not built yet
PASS  no secrets or data in git           clean
PASS  three analysis notebooks            3 present

4/5 checks passed  |  outstanding: revenue reconciles staging = mart


False

---

**Companion documents**

- `docs/Module2_Group7_Development_Plan.pdf` - the plan of record: architecture,
  star schema, quality matrix, phases, ADR-001 to ADR-006.
- `docs/Module2_Group7_Runbook.pdf` - this runbook, print-ready.
- `docs/adr/` - the six architecture decision records, each with the rejected
  alternative and the condition that would make us revisit it.
- `docs/metric_dictionary.md` - every KPI definition, in one place.
- `docs/data_dictionary.md` - every column, and the twelve known data issues.